# Epic 4 Demo — Indicators and Scenario Comparison

**What is this?** In Epic 1 we built the data/computation foundation. In Epic 2 we built scenario templates and workflow configs. In Epic 3 we built the dynamic orchestrator that runs multi-year simulations and produces panel outputs.

Epic 4 turns those raw panel outputs into **decision-ready indicators**. This notebook walks through it step by step:

1. **Distributional indicators** — who bears the burden? Income-decile breakdown of any numeric field
2. **Geographic indicators** — where is the impact? Region-level aggregation with reference validation
3. **Welfare indicators** — who wins, who loses? Baseline vs reform comparison by decile or region
4. **Fiscal indicators** — what does it cost? Revenue, cost, balance, and cumulative tracking
5. **Scenario comparison tables** — side-by-side comparison across multiple scenarios with delta columns
6. **Custom derived indicator formulas** — define your own metrics with a safe expression DSL

In [2]:
from __future__ import annotations

from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
if not (NOTEBOOK_DIR / "data").exists() and (NOTEBOOK_DIR / "notebooks" / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pyarrow as pa

from reformlab.orchestrator.panel import PanelOutput
from reformlab.indicators import (
    # Main computation functions
    compute_distributional_indicators,
    compute_geographic_indicators,
    compute_welfare_indicators,
    compute_fiscal_indicators,
    compare_scenarios,
    apply_custom_formula,
    apply_custom_formulas,
    # Config types
    DistributionalConfig,
    GeographicConfig,
    WelfareConfig,
    FiscalConfig,
    ComparisonConfig,
    CustomFormulaConfig,
    # Result types
    IndicatorResult,
    ComparisonResult,
    ScenarioInput,
    # Exception
    FormulaValidationError,
)

DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = DATA_DIR / "epic4_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def show(table: pa.Table, n: int = 12) -> None:
    """Print a readable table preview."""
    sliced = table.slice(0, n) if table.num_rows > n else table
    cols = sliced.column_names
    rows = [
        [str(sliced.column(c)[i].as_py()) for c in cols]
        for i in range(sliced.num_rows)
    ]
    widths = [max(len(c), *(len(r[j]) for r in rows)) for j, c in enumerate(cols)] if rows else [len(c) for c in cols]
    header = "  ".join(c.ljust(w) for c, w in zip(cols, widths))
    sep = "  ".join("-" * w for w in widths)
    if cols:
        print(header)
        print(sep)
    for row in rows:
        print("  ".join(v.ljust(w) for v, w in zip(row, widths)))
    if table.num_rows > n:
        print(f"  ... ({table.num_rows - n} more rows)")


print(f"Repo root:         {REPO_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Output directory:   {OUTPUT_DIR}")

Repo root:         /Users/lucas/Workspace/reformlab
Notebook directory: /Users/lucas/Workspace/reformlab/notebooks
Output directory:   /Users/lucas/Workspace/reformlab/notebooks/data/epic4_outputs


---
## Setup: Building Demo Panel Data

Indicators consume `PanelOutput` tables — the household-by-year tables produced by the orchestrator in Epic 3.

For this demo we build two panels directly:
- **Baseline** — a carbon tax at EUR 44.60/tCO2 with no redistribution
- **Reform** — same carbon tax but with a progressive lump-sum dividend that benefits lower-income households

Each panel has 100 households across 3 years (2026–2028), with income, region, tax, subsidy, and disposable income fields.

In [3]:
import random

random.seed(42)  # Deterministic demo data

NUM_HOUSEHOLDS = 100
YEARS = [2026, 2027, 2028]
REGIONS = ["11", "24", "27", "28", "32", "44", "52", "53", "75", "76", "84", "93", "94"]
REGION_NAMES = {
    "11": "Île-de-France", "24": "Centre-Val de Loire", "27": "Bourgogne-Franche-Comté",
    "28": "Normandie", "32": "Hauts-de-France", "44": "Grand Est",
    "52": "Pays de la Loire", "53": "Bretagne", "75": "Nouvelle-Aquitaine",
    "76": "Occitanie", "84": "Auvergne-Rhône-Alpes", "93": "Provence-Alpes-Côte d'Azur",
    "94": "Corse",
}

# Assign each household a fixed income and region
hh_incomes = [15000.0 + i * 850.0 + random.gauss(0, 2000) for i in range(NUM_HOUSEHOLDS)]
hh_regions = [random.choice(REGIONS) for _ in range(NUM_HOUSEHOLDS)]


def build_panel(label: str, subsidy_multiplier: float = 0.0) -> PanelOutput:
    """Build a demo panel with carbon tax and optional progressive subsidy."""
    household_ids, year_col = [], []
    incomes, carbon_taxes, subsidies, disposable_incomes = [], [], [], []
    region_codes = []

    for year in YEARS:
        year_offset = (year - 2026)
        for hh_id in range(NUM_HOUSEHOLDS):
            income = hh_incomes[hh_id] * (1 + 0.02 * year_offset)  # 2% annual growth
            # Carbon tax: regressive — roughly flat per household (EUR 150–300)
            carbon_tax = 180.0 + random.gauss(0, 30) + 10 * year_offset
            # Progressive subsidy: inversely proportional to income
            if subsidy_multiplier > 0:
                median_income = 55000.0
                subsidy = max(0.0, subsidy_multiplier * (median_income - income) / median_income * 250.0)
            else:
                subsidy = 0.0

            household_ids.append(hh_id)
            year_col.append(year)
            incomes.append(round(income, 2))
            region_codes.append(hh_regions[hh_id])
            carbon_taxes.append(round(max(carbon_tax, 50.0), 2))  # Floor at EUR 50
            subsidies.append(round(subsidy, 2))
            disposable_incomes.append(round(income - carbon_tax + subsidy, 2))

    table = pa.table(
        {
            "household_id": pa.array(household_ids, type=pa.int64()),
            "year": pa.array(year_col, type=pa.int64()),
            "income": pa.array(incomes, type=pa.float64()),
            "region_code": pa.array(region_codes, type=pa.utf8()),
            "carbon_tax": pa.array(carbon_taxes, type=pa.float64()),
            "subsidy": pa.array(subsidies, type=pa.float64()),
            "disposable_income": pa.array(disposable_incomes, type=pa.float64()),
        }
    )
    return PanelOutput(
        table=table,
        metadata={"label": label, "start_year": 2026, "end_year": 2028, "seed": 42},
    )


baseline_panel = build_panel("baseline", subsidy_multiplier=0.0)
reform_panel = build_panel("reform", subsidy_multiplier=1.5)

print(f"Baseline panel: {baseline_panel.shape[0]} rows, {baseline_panel.shape[1]} columns")
print(f"Reform panel:   {reform_panel.shape[0]} rows, {reform_panel.shape[1]} columns")
print(f"\nColumns: {baseline_panel.table.column_names}")
print(f"Years:   {sorted(set(baseline_panel.table.column('year').to_pylist()))}")
print(f"Regions: {sorted(set(baseline_panel.table.column('region_code').to_pylist()))}")
print("\nBaseline sample (first 8 rows):")
show(baseline_panel.table, n=8)

Baseline panel: 300 rows, 7 columns
Reform panel:   300 rows, 7 columns

Columns: ['household_id', 'year', 'income', 'region_code', 'carbon_tax', 'subsidy', 'disposable_income']
Years:   [2026, 2027, 2028]
Regions: ['11', '24', '27', '28', '32', '44', '52', '53', '75', '76', '84', '93', '94']

Baseline sample (first 8 rows):
household_id  year  income    region_code  carbon_tax  subsidy  disposable_income
------------  ----  --------  -----------  ----------  -------  -----------------
0             2026  14711.82  11           188.6       0.0      14523.22         
1             2026  15504.19  84           244.58      0.0      15259.61         
2             2026  16477.37  93           187.31      0.0      16290.06         
3             2026  18953.97  24           171.12      0.0      18782.85         
4             2026  18144.82  84           183.36      0.0      17961.46         
5             2026  16255.29  75           224.48      0.0      16030.81         
6             202

---
## 1. Distributional Indicators by Income Decile

The most fundamental policy question: **who bears the burden?**

Distributional indicators assign each household to an income decile (1 = lowest 10%, 10 = highest 10%) and compute statistics (count, mean, median, sum, min, max) for every numeric field within each decile.

This reveals whether a policy is progressive (heavier on the rich) or regressive (heavier on the poor).

In [4]:
# Default config: aggregate across all years, group by decile only
dist_result = compute_distributional_indicators(baseline_panel)

print(f"Indicators computed: {len(dist_result.indicators)}")
print(f"Excluded (null income): {dist_result.excluded_count}")
print(f"Warnings: {dist_result.warnings}")
print(f"\nMetadata keys: {sorted(dist_result.metadata.keys())}")

# Show the long-form table — one row per (field, decile, metric)
dist_table = dist_result.to_table()
print(f"\nLong-form table shape: {dist_table.num_rows} rows x {dist_table.num_columns} cols")
print(f"Columns: {dist_table.column_names}")

# Show carbon_tax mean by decile — the key regressivity indicator
print("\nCarbon tax mean by decile (is the tax regressive?):")
for i in range(dist_table.num_rows):
    if (
        dist_table.column("field_name")[i].as_py() == "carbon_tax"
        and dist_table.column("metric")[i].as_py() == "mean"
    ):
        decile = dist_table.column("decile")[i].as_py()
        value = dist_table.column("value")[i].as_py()
        bar = "#" * int(value / 10)
        print(f"  Decile {decile:2d}: EUR {value:8.2f}  {bar}")

Indicators computed: 30
Excluded (null income): 0
Warnings: []

Metadata keys: ['aggregate_years', 'by_year', 'excluded_count', 'group_by_year', 'income_field', 'numeric_fields', 'panel_metadata', 'panel_shape']

Long-form table shape: 180 rows x 5 cols
Columns: ['field_name', 'decile', 'year', 'metric', 'value']

Carbon tax mean by decile (is the tax regressive?):
  Decile  1: EUR   190.10  ###################
  Decile  2: EUR   181.75  ##################
  Decile  3: EUR   191.64  ###################
  Decile  4: EUR   195.93  ###################
  Decile  5: EUR   188.83  ##################
  Decile  6: EUR   194.97  ###################
  Decile  7: EUR   190.28  ###################
  Decile  8: EUR   189.76  ##################
  Decile  9: EUR   186.89  ##################
  Decile 10: EUR   198.13  ###################


In [5]:
# Year-by-year mode: see how the distribution evolves over time
dist_by_year = compute_distributional_indicators(
    baseline_panel,
    DistributionalConfig(by_year=True),
)

by_year_table = dist_by_year.to_table()
print(f"Year-by-year table: {by_year_table.num_rows} rows")

# Show carbon tax mean for decile 1 (poorest) across years
print("\nCarbon tax burden on decile 1 (poorest 10%) over time:")
for i in range(by_year_table.num_rows):
    if (
        by_year_table.column("field_name")[i].as_py() == "carbon_tax"
        and by_year_table.column("metric")[i].as_py() == "mean"
        and by_year_table.column("decile")[i].as_py() == 1
    ):
        year = by_year_table.column("year")[i].as_py()
        value = by_year_table.column("value")[i].as_py()
        print(f"  {year}: EUR {value:.2f}")

Year-by-year table: 540 rows

Carbon tax burden on decile 1 (poorest 10%) over time:
  2026: EUR 195.91
  2027: EUR 174.11
  2028: EUR 200.29


---
## 2. Geographic Aggregation Indicators

Policy impacts vary across regions — a carbon tax hits rural areas with longer commutes differently from dense urban centres.

Geographic indicators group households by region code and compute the same statistics. An optional **reference table** validates region codes: unknown codes are bucketed into `"_UNMATCHED"` so nothing is silently dropped.

In [6]:
# Build a reference table of valid French region codes
ref_table = pa.table(
    {
        "region_code": pa.array(list(REGION_NAMES.keys()), type=pa.utf8()),
        "region_name": pa.array(list(REGION_NAMES.values()), type=pa.utf8()),
    }
)

geo_result = compute_geographic_indicators(
    baseline_panel,
    GeographicConfig(reference_table=ref_table),
)

print(f"Indicators computed: {len(geo_result.indicators)}")
print(f"Excluded (null region): {geo_result.excluded_count}")
print(f"Unmatched regions: {geo_result.unmatched_count}")

# Show mean carbon tax by region
geo_table = geo_result.to_table()
print(f"\nCarbon tax mean by region:")
region_means = []
for i in range(geo_table.num_rows):
    if (
        geo_table.column("field_name")[i].as_py() == "carbon_tax"
        and geo_table.column("metric")[i].as_py() == "mean"
    ):
        region = geo_table.column("region")[i].as_py()
        value = geo_table.column("value")[i].as_py()
        name = REGION_NAMES.get(region, region)
        region_means.append((name, value))

for name, value in sorted(region_means, key=lambda x: -x[1]):
    bar = "#" * int(value / 10)
    print(f"  {name:35s} EUR {value:8.2f}  {bar}")

Indicators computed: 52
Excluded (null region): 0
Unmatched regions: 0

Carbon tax mean by region:
  Corse                               EUR   204.26  ####################
  Nouvelle-Aquitaine                  EUR   198.85  ###################
  Provence-Alpes-Côte d'Azur          EUR   197.93  ###################
  Pays de la Loire                    EUR   195.49  ###################
  Normandie                           EUR   194.68  ###################
  Centre-Val de Loire                 EUR   190.54  ###################
  Bretagne                            EUR   189.41  ##################
  Île-de-France                       EUR   187.42  ##################
  Occitanie                           EUR   186.44  ##################
  Auvergne-Rhône-Alpes                EUR   184.21  ##################
  Hauts-de-France                     EUR   182.71  ##################
  Bourgogne-Franche-Comté             EUR   181.43  ##################
  Grand Est                           EUR 

In [7]:
# Year-by-year geographic breakdown
geo_by_year = compute_geographic_indicators(
    baseline_panel,
    GeographicConfig(reference_table=ref_table, by_year=True),
)

geo_year_table = geo_by_year.to_table()
print(f"Year-by-year geographic table: {geo_year_table.num_rows} rows")

# Show Île-de-France carbon tax mean over time
print("\nÎle-de-France (region 11) carbon tax mean over time:")
for i in range(geo_year_table.num_rows):
    if (
        geo_year_table.column("field_name")[i].as_py() == "carbon_tax"
        and geo_year_table.column("metric")[i].as_py() == "mean"
        and geo_year_table.column("region")[i].as_py() == "11"
    ):
        year = geo_year_table.column("year")[i].as_py()
        value = geo_year_table.column("value")[i].as_py()
        print(f"  {year}: EUR {value:.2f}")

Year-by-year geographic table: 936 rows

Île-de-France (region 11) carbon tax mean over time:
  2026: EUR 184.16
  2027: EUR 190.45
  2028: EUR 187.64


---
## 3. Welfare Indicators (Baseline vs Reform)

The core policy design question: **who wins and who loses under the reform?**

Welfare indicators join baseline and reform panels on `(household_id, year)`, compute `net_change = reform - baseline` for a welfare field (e.g., disposable income), and classify each household as a **winner**, **loser**, or **neutral**.

Results are grouped by income decile (default) or region.

In [8]:
# Default: group by income decile, aggregate across years
welfare_result = compute_welfare_indicators(
    baseline_panel,
    reform_panel,
    WelfareConfig(welfare_field="disposable_income", threshold=100.0),
)

print(f"Indicators computed: {len(welfare_result.indicators)}")
print(f"Excluded (null income): {welfare_result.excluded_count}")
print(f"Unmatched households: {welfare_result.unmatched_count}")

# Show winner/loser breakdown by decile
print("\nWinner/Loser analysis by income decile (threshold: EUR 100):")
print(f"{'Decile':>8s}  {'Winners':>8s}  {'Losers':>8s}  {'Neutral':>8s}  {'Net Change':>12s}")
print("-" * 55)

welfare_table = welfare_result.to_table()
# Collect per-decile summary
decile_data: dict[int, dict[str, float]] = {}
for i in range(welfare_table.num_rows):
    decile = welfare_table.column("decile")[i].as_py()
    metric = welfare_table.column("metric")[i].as_py()
    value = welfare_table.column("value")[i].as_py()
    decile_data.setdefault(decile, {})[metric] = value

for decile in sorted(decile_data):
    d = decile_data[decile]
    print(
        f"{decile:8d}  {d.get('winner_count', 0):8.0f}  {d.get('loser_count', 0):8.0f}  "
        f"{d.get('neutral_count', 0):8.0f}  EUR {d.get('net_change', 0):9.2f}"
    )

Indicators computed: 10
Excluded (null income): 0
Unmatched households: 0

Winner/Loser analysis by income decile (threshold: EUR 100):
  Decile   Winners    Losers   Neutral    Net Change
-------------------------------------------------------
       1        30         0         0  EUR   7516.42
       2        29         0         1  EUR   5540.82
       3        23         0         7  EUR   3993.45
       4         9         0        21  EUR   2422.76
       5         1         0        29  EUR    299.89
       6         0         1        29  EUR    262.49
       7         0         1        29  EUR    -58.11
       8         0         0        30  EUR    -62.63
       9         0         0        30  EUR     57.83
      10         0         1        29  EUR     14.65


In [9]:
# Group by region instead of decile
welfare_by_region = compute_welfare_indicators(
    baseline_panel,
    reform_panel,
    WelfareConfig(
        welfare_field="disposable_income",
        threshold=100.0,
        group_by_decile=False,
        group_by_region=True,
    ),
)

welfare_region_table = welfare_by_region.to_table()
print("Winner/Loser analysis by region:")
print(f"{'Region':>35s}  {'Winners':>8s}  {'Losers':>8s}  {'Net EUR':>12s}")
print("-" * 72)

region_welfare: dict[str, dict[str, float]] = {}
for i in range(welfare_region_table.num_rows):
    region = welfare_region_table.column("region")[i].as_py()
    metric = welfare_region_table.column("metric")[i].as_py()
    value = welfare_region_table.column("value")[i].as_py()
    region_welfare.setdefault(region, {})[metric] = value

for region in sorted(region_welfare, key=lambda r: -region_welfare[r].get("net_change", 0)):
    d = region_welfare[region]
    name = REGION_NAMES.get(region, region)
    print(
        f"{name:>35s}  {d.get('winner_count', 0):8.0f}  {d.get('loser_count', 0):8.0f}  "
        f"EUR {d.get('net_change', 0):9.2f}"
    )

Winner/Loser analysis by region:
                             Region   Winners    Losers       Net EUR
------------------------------------------------------------------------
               Auvergne-Rhône-Alpes        13         0  EUR   2771.65
                 Nouvelle-Aquitaine        12         0  EUR   2675.07
                              Corse        11         1  EUR   2618.72
                    Hauts-de-France        10         0  EUR   2082.49
         Provence-Alpes-Côte d'Azur         9         0  EUR   2029.34
                Centre-Val de Loire         9         0  EUR   1957.28
            Bourgogne-Franche-Comté         5         0  EUR   1405.37
                      Île-de-France         7         1  EUR   1377.96
                          Grand Est         6         0  EUR   1106.39
                           Bretagne         3         1  EUR    581.62
                   Pays de la Loire         3         0  EUR    542.45
                          Occitanie        

---
## 4. Fiscal Indicators (Revenue, Cost, Balance)

For budget analysis: **how much does the policy collect, how much does it spend, and what's the balance?**

Fiscal indicators sum configured revenue and cost fields across all households, optionally year-by-year with running cumulative totals.

In [10]:
# Fiscal indicators for the reform panel (has both carbon_tax revenue and subsidy cost)
fiscal_result = compute_fiscal_indicators(
    reform_panel,
    FiscalConfig(
        revenue_fields=["carbon_tax"],
        cost_fields=["subsidy"],
        by_year=True,
        cumulative=True,
    ),
)

print(f"Indicators computed: {len(fiscal_result.indicators)}")
print(f"Warnings: {fiscal_result.warnings}")

fiscal_table = fiscal_result.to_table()
print(f"\nFiscal table: {fiscal_table.num_rows} rows")
print("\nYear-by-year fiscal summary:")

# Parse the long-form table into a readable summary
year_fiscal: dict[int, dict[str, float]] = {}
for i in range(fiscal_table.num_rows):
    year = fiscal_table.column("year")[i].as_py()
    metric = fiscal_table.column("metric")[i].as_py()
    value = fiscal_table.column("value")[i].as_py()
    year_fiscal.setdefault(year, {})[metric] = value

print(f"{'Year':>6s}  {'Revenue':>12s}  {'Cost':>12s}  {'Balance':>12s}  {'Cum.Balance':>12s}")
print("-" * 62)
for year in sorted(year_fiscal):
    d = year_fiscal[year]
    print(
        f"{year:6d}  EUR {d.get('revenue', 0):9.0f}  EUR {d.get('cost', 0):9.0f}  "
        f"EUR {d.get('balance', 0):9.0f}  EUR {d.get('cumulative_balance', 0):9.0f}"
    )

Indicators computed: 3
Warnings: []

Fiscal table: 18 rows

Year-by-year fiscal summary:
  Year       Revenue          Cost       Balance   Cum.Balance
--------------------------------------------------------------
  2026  EUR     18019  EUR      6587  EUR     11432  EUR     11432
  2027  EUR     18806  EUR      6371  EUR     12435  EUR     23867
  2028  EUR     19566  EUR      6172  EUR     13394  EUR     37260


In [11]:
# Compare fiscal position: baseline (no subsidy) vs reform (with subsidy)
fiscal_baseline = compute_fiscal_indicators(
    baseline_panel,
    FiscalConfig(
        revenue_fields=["carbon_tax"],
        cost_fields=["subsidy"],
        by_year=True,
        cumulative=True,
    ),
)

print("Baseline fiscal (no redistribution):")
base_fiscal_table = fiscal_baseline.to_table()
for i in range(base_fiscal_table.num_rows):
    if base_fiscal_table.column("metric")[i].as_py() == "balance":
        year = base_fiscal_table.column("year")[i].as_py()
        value = base_fiscal_table.column("value")[i].as_py()
        print(f"  {year}: EUR {value:,.0f} (all revenue, no cost)")

print("\nReform fiscal (with progressive dividend):")
for i in range(fiscal_table.num_rows):
    if fiscal_table.column("metric")[i].as_py() == "balance":
        year = fiscal_table.column("year")[i].as_py()
        value = fiscal_table.column("value")[i].as_py()
        print(f"  {year}: EUR {value:,.0f} (revenue minus subsidy cost)")

Baseline fiscal (no redistribution):
  2026: EUR 18,509 (all revenue, no cost)
  2027: EUR 18,971 (all revenue, no cost)
  2028: EUR 19,768 (all revenue, no cost)

Reform fiscal (with progressive dividend):
  2026: EUR 11,432 (revenue minus subsidy cost)
  2027: EUR 12,435 (revenue minus subsidy cost)
  2028: EUR 13,394 (revenue minus subsidy cost)


---
## 5. Scenario Comparison Tables

When you have indicators from multiple scenarios, you want to put them **side by side** and see the deltas.

`compare_scenarios` does exactly that:
- Takes 2+ `ScenarioInput` wrappers (label + `IndicatorResult`)
- Joins them on their natural keys (field_name, decile/region, year, metric)
- Adds `delta_*` columns (absolute change from baseline) and `pct_delta_*` columns (percentage change)
- Exports to CSV or Parquet

In [12]:
# Compute distributional indicators for both scenarios
dist_baseline = compute_distributional_indicators(baseline_panel)
dist_reform = compute_distributional_indicators(reform_panel)

# Compare them side-by-side
comparison = compare_scenarios(
    [
        ScenarioInput(label="baseline", indicators=dist_baseline),
        ScenarioInput(label="reform", indicators=dist_reform),
    ],
    ComparisonConfig(baseline_label="baseline", include_deltas=True, include_pct_deltas=True),
)

print(f"Comparison table shape: {comparison.table.num_rows} rows x {comparison.table.num_columns} cols")
print(f"Columns: {comparison.table.column_names}")
print(f"Warnings: {comparison.warnings}")
print(f"\nMetadata:")
print(f"  Schema: {comparison.metadata.get('indicator_schema')}")
print(f"  Labels: {comparison.metadata.get('scenario_labels')}")
print(f"  Join keys: {comparison.metadata.get('join_keys')}")

# Show subsidy mean by decile — the reform redistributes, so this should differ
print("\nSubsidy mean by decile — baseline vs reform:")
ct = comparison.table
for i in range(ct.num_rows):
    if (
        ct.column("field_name")[i].as_py() == "subsidy"
        and ct.column("metric")[i].as_py() == "mean"
    ):
        decile = ct.column("decile")[i].as_py()
        base_val = ct.column("baseline")[i].as_py()
        reform_val = ct.column("reform")[i].as_py()
        delta = ct.column("delta_reform")[i].as_py()
        print(
            f"  Decile {decile:2d}: baseline={base_val:8.2f}  reform={reform_val:8.2f}  "
            f"delta={delta:+9.2f}"
        )

Comparison table shape: 180 rows x 8 cols
Columns: ['field_name', 'decile', 'year', 'metric', 'baseline', 'reform', 'delta_reform', 'pct_delta_reform']
Warnings: []

Metadata:
  Schema: decile
  Labels: ['baseline', 'reform']
  Join keys: ['field_name', 'decile', 'year', 'metric']

Subsidy mean by decile — baseline vs reform:
  Decile  1: baseline=    0.00  reform=  246.01  delta=  +246.01
  Decile  2: baseline=    0.00  reform=  183.82  delta=  +183.82
  Decile  3: baseline=    0.00  reform=  126.91  delta=  +126.91
  Decile  4: baseline=    0.00  reform=   71.42  delta=   +71.42
  Decile  5: baseline=    0.00  reform=    9.50  delta=    +9.50
  Decile  6: baseline=    0.00  reform=    0.00  delta=    +0.00
  Decile  7: baseline=    0.00  reform=    0.00  delta=    +0.00
  Decile  8: baseline=    0.00  reform=    0.00  delta=    +0.00
  Decile  9: baseline=    0.00  reform=    0.00  delta=    +0.00
  Decile 10: baseline=    0.00  reform=    0.00  delta=    +0.00


In [13]:
# Three-way comparison: baseline, reform, and a "high ambition" variant
high_ambition_panel = build_panel("high_ambition", subsidy_multiplier=2.5)
dist_high = compute_distributional_indicators(high_ambition_panel)

comparison_3way = compare_scenarios(
    [
        ScenarioInput(label="baseline", indicators=dist_baseline),
        ScenarioInput(label="reform", indicators=dist_reform),
        ScenarioInput(label="high_ambition", indicators=dist_high),
    ],
)

print(f"3-way comparison: {comparison_3way.table.num_rows} rows x {comparison_3way.table.num_columns} cols")
print(f"Columns: {comparison_3way.table.column_names}")

# Export to CSV and Parquet
comparison.export_csv(OUTPUT_DIR / "distributional_comparison.csv")
comparison.export_parquet(OUTPUT_DIR / "distributional_comparison.parquet")
comparison_3way.export_csv(OUTPUT_DIR / "distributional_3way_comparison.csv")

print("\nExported comparison files to notebooks/data/epic4_outputs/")

3-way comparison: 180 rows x 11 cols
Columns: ['field_name', 'decile', 'year', 'metric', 'baseline', 'reform', 'high_ambition', 'delta_reform', 'delta_high_ambition', 'pct_delta_reform', 'pct_delta_high_ambition']



Exported comparison files to notebooks/data/epic4_outputs/


In [14]:
# Fiscal comparison across scenarios
fiscal_high = compute_fiscal_indicators(
    high_ambition_panel,
    FiscalConfig(
        revenue_fields=["carbon_tax"],
        cost_fields=["subsidy"],
        by_year=True,
        cumulative=True,
    ),
)

fiscal_comparison = compare_scenarios(
    [
        ScenarioInput(label="baseline", indicators=fiscal_baseline),
        ScenarioInput(label="reform", indicators=fiscal_result),
        ScenarioInput(label="high_ambition", indicators=fiscal_high),
    ],
)

print("Fiscal comparison — balance metric across scenarios:")
fct = fiscal_comparison.table
print(f"{'Year':>6s}  {'Baseline':>12s}  {'Reform':>12s}  {'High Ambition':>14s}")
print("-" * 52)
for i in range(fct.num_rows):
    if fct.column("metric")[i].as_py() == "balance":
        year = fct.column("year")[i].as_py()
        base = fct.column("baseline")[i].as_py()
        ref = fct.column("reform")[i].as_py()
        high = fct.column("high_ambition")[i].as_py()
        print(f"{year:6d}  EUR {base:9.0f}  EUR {ref:9.0f}  EUR {high:11.0f}")

fiscal_comparison.export_csv(OUTPUT_DIR / "fiscal_comparison.csv")
print("\nExported fiscal comparison to notebooks/data/epic4_outputs/")

Fiscal comparison — balance metric across scenarios:
  Year      Baseline        Reform   High Ambition
----------------------------------------------------
  2026  EUR     18509  EUR     11432  EUR        6462
  2027  EUR     18971  EUR     12435  EUR        8070
  2028  EUR     19768  EUR     13394  EUR       10030

Exported fiscal comparison to notebooks/data/epic4_outputs/


---
## 6. Custom Derived Indicator Formulas

Sometimes the built-in metrics aren't enough — you need a custom calculation. For example:
- "Surplus as % of revenue" → `balance / revenue * 100`
- "Average tax per household" → `sum / count`

`apply_custom_formula` parses a safe expression DSL (no `eval`, no code injection) that supports:
- Arithmetic: `+`, `-`, `*`, `/`
- Parentheses for grouping
- Numeric constants
- References to existing metric names

Division by zero returns `null` (not an error). Formulas are tracked in result metadata for governance.

In [15]:
# Start with fiscal indicators — they have revenue, cost, balance metrics
fiscal_for_custom = compute_fiscal_indicators(
    reform_panel,
    FiscalConfig(
        revenue_fields=["carbon_tax"],
        cost_fields=["subsidy"],
        by_year=True,
        cumulative=False,  # Keep it simple for formulas
    ),
)

# Apply a single custom formula: surplus percentage
surplus_formula = CustomFormulaConfig(
    source_field="fiscal_summary",
    output_metric="surplus_pct",
    expression="balance / revenue * 100",
    description="Net fiscal surplus as a percentage of total revenue",
)

with_surplus = apply_custom_formula(fiscal_for_custom, surplus_formula)

# The derived metric appears in the to_table() output
derived_table = with_surplus.to_table()
print("Fiscal indicators with custom surplus_pct metric:")
show(derived_table)

Fiscal indicators with custom surplus_pct metric:
field_name      year  metric       value             
--------------  ----  -----------  ------------------
fiscal_summary  2026  revenue      18018.749999999996
fiscal_summary  2026  cost         6586.86           
fiscal_summary  2026  balance      11431.889999999996
fiscal_summary  2027  revenue      18805.6           
fiscal_summary  2027  cost         6370.61           
fiscal_summary  2027  balance      12434.989999999998
fiscal_summary  2028  revenue      19566.05          
fiscal_summary  2028  cost         6172.460000000002 
fiscal_summary  2028  balance      13393.589999999997
fiscal_summary  2026  surplus_pct  63.44441207075961 
fiscal_summary  2027  surplus_pct  66.12386735865911 
fiscal_summary  2028  surplus_pct  68.45321360213225 


In [16]:
# Chain multiple formulas — later formulas can reference earlier ones
formulas = [
    CustomFormulaConfig(
        source_field="fiscal_summary",
        output_metric="cost_ratio",
        expression="cost / revenue * 100",
        description="Cost as percentage of revenue",
    ),
    CustomFormulaConfig(
        source_field="fiscal_summary",
        output_metric="net_rate",
        expression="balance / revenue",
        description="Net fiscal rate (1 = all revenue retained, 0 = fully redistributed)",
    ),
]

enriched = apply_custom_formulas(fiscal_for_custom, formulas)

enriched_table = enriched.to_table()
print(f"Enriched table: {enriched_table.num_rows} rows (added {enriched_table.num_rows - fiscal_for_custom.to_table().num_rows} derived rows)")

# Show the derived metrics
print("\nDerived metrics by year:")
for i in range(enriched_table.num_rows):
    metric = enriched_table.column("metric")[i].as_py()
    if metric in ("cost_ratio", "net_rate"):
        year = enriched_table.column("year")[i].as_py()
        value = enriched_table.column("value")[i].as_py()
        print(f"  {year}  {metric:12s} = {value:.4f}")

# Metadata tracks all formulas applied
print(f"\nFormula metadata: {enriched.metadata.get('custom_formulas')}")

Enriched table: 15 rows (added 6 derived rows)

Derived metrics by year:
  2026  cost_ratio   = 36.5556
  2027  cost_ratio   = 33.8761
  2028  cost_ratio   = 31.5468
  2026  net_rate     = 0.6344
  2027  net_rate     = 0.6612
  2028  net_rate     = 0.6845

Formula metadata: [{'source_field': 'fiscal_summary', 'output_metric': 'cost_ratio', 'expression': 'cost / revenue * 100', 'description': 'Cost as percentage of revenue'}, {'source_field': 'fiscal_summary', 'output_metric': 'net_rate', 'expression': 'balance / revenue', 'description': 'Net fiscal rate (1 = all revenue retained, 0 = fully redistributed)'}]


In [17]:
# Custom formulas on distributional indicators too
# Example: "average tax per household" from decile-level stats
dist_with_avg = apply_custom_formula(
    dist_baseline,
    CustomFormulaConfig(
        source_field="carbon_tax",
        output_metric="avg_tax",
        expression="sum / count",
        description="Average carbon tax per household in the decile",
    ),
)

avg_table = dist_with_avg.to_table()
print("Average carbon tax per household by decile (via custom formula):")
for i in range(avg_table.num_rows):
    if (
        avg_table.column("field_name")[i].as_py() == "carbon_tax"
        and avg_table.column("metric")[i].as_py() == "avg_tax"
    ):
        decile = avg_table.column("decile")[i].as_py()
        value = avg_table.column("value")[i].as_py()
        print(f"  Decile {decile:2d}: EUR {value:.2f}")

Average carbon tax per household by decile (via custom formula):
  Decile  1: EUR 190.10
  Decile  2: EUR 181.75
  Decile  3: EUR 191.64
  Decile  4: EUR 195.93
  Decile  5: EUR 188.83
  Decile  6: EUR 194.97
  Decile  7: EUR 190.28
  Decile  8: EUR 189.76
  Decile  9: EUR 186.89
  Decile 10: EUR 198.13


In [18]:
# What happens with invalid formulas?
# The parser catches syntax errors and gives helpful messages

bad_formulas = [
    ("", "Empty expression"),
    ("revenue + @invalid", "Unknown character"),
    ("(revenue + cost", "Unbalanced parentheses"),
    ("nonexistent_metric * 2", "Unknown metric reference"),
]

for expr, label in bad_formulas:
    try:
        apply_custom_formula(
            fiscal_for_custom,
            CustomFormulaConfig(
                source_field="fiscal_summary",
                output_metric="bad",
                expression=expr,
            ),
        )
    except (FormulaValidationError, ValueError) as e:
        print(f"  [{label}] {type(e).__name__}: {e}")
        if isinstance(e, FormulaValidationError) and e.suggestion:
            print(f"    Suggestion: {e.suggestion}")

  [Empty expression] FormulaValidationError: Formula expression cannot be empty
  [Unknown character] FormulaValidationError: Invalid character in expression: '@' at position 10
  [Unbalanced parentheses] FormulaValidationError: Expected RPAREN, got 'EOF' at position 15
  [Unknown metric reference] FormulaValidationError: Metric 'nonexistent_metric' not found. Available metrics: balance, cost, revenue


---
## What We Built (Summary)

This EPIC 4 demo covers the full indicator and comparison layer:

| Capability | In plain terms |
|---|---|
| Distributional indicators | Assign households to income deciles and compute statistics for every numeric field — reveals if a policy is progressive or regressive |
| Geographic indicators | Group by region with optional reference validation — shows where the impact lands |
| Welfare indicators | Join baseline and reform panels, classify households as winners/losers/neutral by decile or region |
| Fiscal indicators | Sum revenue and cost fields with optional cumulative tracking — the budget view |
| Scenario comparison tables | Side-by-side comparison of 2+ scenarios with absolute and percentage deltas, CSV/Parquet export |
| Custom derived formulas | Safe expression DSL for computing new metrics from existing ones — no `eval`, no code injection |
| Long-form output | Every indicator type produces a stable `to_table()` output suitable for export and downstream processing |
| Metadata + warnings | Full computation metadata, excluded/unmatched counts, and warning messages for transparency |

**What came before:** Epic 1 (data/computation foundation), Epic 2 (scenario templates + workflow config), Epic 3 (dynamic orchestrator + panel output).

**What comes next:** Governance and reproducibility (Epic 5) will add immutable run manifests, assumption tracking, and lineage graphs. The interface layer (Epic 6) will wrap everything in a stable Python API, notebooks, and no-code GUI.

In [19]:
print("Generated files in notebooks/data/epic4_outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.relative_to(DATA_DIR)}")

Generated files in notebooks/data/epic4_outputs:
  epic4_outputs/distributional_3way_comparison.csv
  epic4_outputs/distributional_comparison.csv
  epic4_outputs/distributional_comparison.parquet
  epic4_outputs/fiscal_comparison.csv
